# Notebook 3 — Logistic regression and decisions

The target changes from a number to a category, and with it the whole evaluation
toolkit. This notebook ends with the exercise that matters most in practice: choosing
a threshold when the two errors do not cost the same.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix,
                             precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams['figure.figsize'] = (8, 4)

from pathlib import Path

# works whether you run this from the notebooks folder or from a subfolder
DATA = Path('data') if Path('data').exists() else Path('..') / 'data'

weekly = pd.read_csv(DATA / 'bacterial_spot_weekly.csv')
weekly.head()

---
## 1. The target, and how rare it is

In [ ]:
NUM = ['wet_hours_week', 'night_temp_c', 'rain_mm_week',
       'inoculum_log', 'days_after_transplant']
CAT = ['cultivar']

X = weekly[NUM + CAT]
t = weekly['infection_event']

prevalence = t.mean()
print(f'plot-weeks           : {len(t):,}')
print(f'infection events     : {t.sum():,}')
print(f'prevalence           : {prevalence:.3f}')
print(f'"never an event" accuracy: {1 - prevalence:.3f}')

Write that last number down. Any classifier that cannot beat **85.7% accuracy** has
learned nothing at all, because predicting "no event" every single week already scores
that.

---
## 2. Fit the model

We fit on the **raw, unscaled** features here. Scaling would be fine for prediction,
but leaving the units alone means the coefficients can be read directly as odds
ratios — which is what makes this model usable in extension.

In [ ]:
pre = ColumnTransformer([
    ('num', 'passthrough', NUM),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CAT),
])

clf = Pipeline([('pre', pre), ('m', LogisticRegression(max_iter=2000))])
clf.fit(X, t)

names = NUM + list(clf.named_steps['pre']
                   .named_transformers_['cat'].get_feature_names_out(CAT))
coefs = clf.named_steps['m'].coef_[#]
intercept = clf.named_steps['m'].intercept_[0]

print(f'intercept {intercept:.3f}')
pd.Series(coefs, index=names).round(4)

---
## 3. Turn weights into odds ratios

The linear part of a logistic regression **is** the log-odds. Exponentiate a weight and
you get a multiplier on the odds — the language epidemiologists already speak.

In [ ]:
odds = pd.Series(np.exp(#), index=names)

print(f'+1 h of leaf wetness       : x {odds["wet_hours_week"]:.3f}')
print(f'+10 h of leaf wetness      : x {np.exp(10 * coefs[0]):.2f}')
print(f'+2 C warmer nights         : x {np.exp(#):.2f}')
print(f'+1 log unit of inoculum    : x {odds["inoculum_log"]:.2f}')
print(f'cv. Tygress vs FL-8000     : x {odds["cultivar_Tygress"]:.2f}')

Read the last one out loud: planting Tygress instead of FL-8000 cuts the odds of an
infection event to roughly a quarter, holding the weather and inoculum fixed.

That is a sentence a grower can act on.

---
## 4. One plot-week, by hand

Prediction is just arithmetic with frozen weights. Do it once manually so nobody
thinks a trained model is doing something mysterious at prediction time.

In [ ]:
example = pd.DataFrame([{
    'wet_hours_week': 46.0,
    'night_temp_c': 21.5,
    'rain_mm_week': 22.0,
    'inoculum_log': 3.4,
    'days_after_transplant': 49,
    'cultivar': 'FL-8000',
}])

z = clf.decision_function(example)[0]
p = 1 / (1 + np.exp(-z))

print(f'linear part  z = {z:+.3f}')
print(f'probability  P = {p:.3f}')
print(f'sklearn says P = {clf.predict_proba(example)[0, 1]:.3f}')

---
## 5. The decision boundary

Hold everything else fixed and sweep two features. This picture is the same object as
the infection-period charts in an extension handbook — a partition of a
weather space into act / do-not-act.

In [ ]:
gw = np.linspace(2, 80, 150)
gt = np.linspace(12, 27, 150)
GW, GT = np.meshgrid(gw, gt)

grid = pd.DataFrame({
    'wet_hours_week': GW.ravel(),
    'night_temp_c': GT.ravel(),
    'rain_mm_week': weekly.rain_mm_week.median(),
    'inoculum_log': weekly.inoculum_log.median(),
    'days_after_transplant': 42,
    'cultivar': 'FL-8000',
})

P = clf.predict_proba(grid)[:, #].reshape(GW.shape)

fig, ax = plt.subplots(figsize=(7, 4))
cs = ax.contourf(GW, GT, P, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.85)
ax.contour(GW, GT, P, levels=[#], colors='black', linewidths=3)
fig.colorbar(cs, ax=ax, label='P(infection)')
ax.set_xlabel('Leaf wetness this week (h)')
ax.set_ylabel('Night temperature (C)')
plt.tight_layout()

**Try it**: change `cultivar` to `'Tygress'` and re-run. Watch the black line move.
That shift is the resistance effect, in the units a spray decision is made in.

---
## 6. Honest predictions

Everything from here on uses **leave-one-year-out** cross-validated probabilities, so
no plot-week is ever scored by a model that saw its season.

In [ ]:
proba = cross_val_predict(clf, X, t, cv=GroupKFold(10), groups=#,
                          method='predict_proba')[:, 1]

print('cross-validated probabilities for', len(proba), 'plot-weeks')
print(f'AUC {roc_auc_score(t, proba):.3f}')

---
## 7. Accuracy, and why it lies

In [ ]:
pred_050 = (proba >= 0.5).astype(int)

accuracy = (pred_050 == t).mean()
print(f'our model, threshold 0.5 : accuracy {accuracy:.3f}')
print(f'"never an event"         : accuracy {1 - prevalence:.3f}')
print()
print(f'infection events caught  : {pred_050[t == 1].sum()} of {t.sum()}'
      f'  ({recall_score(t, pred_050):.1%})')

Two models with effectively the same accuracy. One of them has no features at all.

The one with features catches **9%** of the infection events that happened.

---
## 8. The confusion matrix

In [ ]:
cm = confusion_matrix(t, pred_050)
tn, fp, fn, tp = cm.ravel()

print(f'  true positives  {tp:5d}   caught it')
print(f'  false negatives {fn:5d}   missed epidemic')
print(f'  false positives {fp:5d}   wasted spray')
print(f'  true negatives  {tn:5d}   correct')

ConfusionMatrixDisplay(cm, display_labels=['no event', 'event']).plot(
    cmap='Blues', colorbar=False)
plt.tight_layout()

---
## 9. Sweep the threshold

Now the central exercise of the section. Compute the metrics across a range of
cut-points and watch which ones move.

In [ ]:
def metrics_at(thr):
    yhat = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(t, yhat).ravel()
    return {
        'threshold': thr,
        'recall': recall_score(t, yhat, zero_division=0),
        'specificity': #,
        'precision': precision_score(t, yhat, zero_division=0),
        'accuracy': #,
        'alerts': int(tp + fp),
    }

sweep = pd.DataFrame([metrics_at(x) for x in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]])
sweep.round(3)

### Read the table before you go on

Accuracy is highest at the top and falls as you move down. Recall does the opposite.

The metric that gets worse is the one that never mattered.

In [ ]:
prec, rec, thr = precision_recall_curve(t, proba)
fpr, tpr, _ = roc_curve(t, proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(thr, prec[:-1], color='#3677A5', lw=2.5, label='Precision')
axes[0].plot(thr, rec[:-1], color='#FF4A00', lw=2.5, label='Recall')
axes[0].axvline(0.5, color='grey', ls=':', lw=2)
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Score')
axes[0].set_xlim(0, 0.9); axes[0].legend(frameon=False)

axes[1].plot(fpr, tpr, color='#FF4A00', lw=2.5)
axes[1].plot([0, 1], [0, 1], '--', color='grey', lw=2)
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title(f'AUC = {roc_auc_score(t, proba):.3f}')

plt.tight_layout()

---
## 10. Group exercise — choose your threshold

Your instructor will assign your group a scenario. Each has a different **cost ratio**:
how many unnecessary sprays you would accept to avoid one missed infection period.

| Scenario | Cost ratio (FN : FP) |
|---|---|
| A — cheap protectant, routine disease | 3 : 1 |
| B — expensive product, resistance risk | 1 : 1 |
| C — seed crop, zero tolerance for loss | 20 : 1 |
| D — quarantine pathogen surveillance | 200 : 1 |

Compute the total cost across all plot-weeks for each threshold and find the minimum.

In [ ]:
def total_cost(thr, fn_cost, fp_cost=1.0):
    yhat = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(t, yhat).ravel()
    return #

thresholds = np.arange(0.02, 0.90, 0.01)

for label, ratio in [('A', 3), ('B', 1), ('C', 20), ('D', 200)]:
    costs = [total_cost(x, ratio) for x in thresholds]
    best = thresholds[int(np.argmin(costs))]
    m = metrics_at(best)
    print(f'scenario {label} (FN:FP = {ratio:3d}:1)  best threshold {best:.2f}'
          f'   recall {m["recall"]:.2f}   precision {m["precision"]:.2f}'
          f'   alerts {m["alerts"]:,}')

### One result is worth pausing on

At 200 : 1 the optimal threshold drops to about 0.02 and the model flags nearly every
plot-week. When the cost of missing an event is that extreme, the arithmetic stops
recommending a model and starts recommending "act every time".

That is not a failure of the code. It is the honest answer, and it tells you something
real: for a genuine zero-tolerance pathogen, a prediction tool with this much
uncertainty cannot improve on blanket action. Knowing when *not* to deploy a model is
part of the job.

### Report back to the room

* What threshold did your scenario give you?
* How many alerts per season does that mean for a grower with 40 plots?
* At what point does the number of false alarms destroy trust in the tool, regardless
  of what the cost arithmetic says?

That last question is not answerable from the data, and it is the one that decides
whether a decision support tool gets used.

---
## 11. More than two classes

The same model handles several outcomes at once. We do not have a multi-pathogen
target here, so we build one: severity bands from the season data, which is a
realistic triage question — is this field low, moderate or high risk?

In [ ]:
season = pd.read_csv(DATA / 'bacterial_spot_season.csv')
season['band'] = pd.cut(season.final_severity, bins=[-0.1, 30, 65, 100.1],
                        labels=['low', 'moderate', 'high'])

SNUM = ['wetness_hours', 'mean_temp_c', 'rain_mm', 'inoculum_log']
Xs = season[SNUM + ['cultivar']].fillna(season.inoculum_log.median())
ts = season['band']

multi = Pipeline([
    ('pre', ColumnTransformer([
        ('num', StandardScaler(), SNUM),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['cultivar']),
    ])),
    ('m', LogisticRegression(max_iter=2000)),
])

acc = cross_val_score(multi, Xs, ts, cv=GroupKFold(10), groups=season.year).mean()
print(f'3-class accuracy, leaving out seasons: {acc:.3f}')
print(f'always guessing the commonest band:    {ts.value_counts(normalize=True).max():.3f}')

In [ ]:
multi.fit(Xs, ts)
probs = multi.predict_proba(Xs[:5])

pd.DataFrame(probs, columns=multi.classes_).round(3)

Each row sums to one — that is softmax. The model spreads a single unit of belief
across the three bands rather than fitting three separate models.

---
## 12. What to take from this notebook

* Prevalence sets the accuracy you must beat before you have done anything.
* Exponentiated weights are odds ratios, and they are how you explain the model.
* Cross-validated probabilities, grouped by season, are the only honest ones here.
* The threshold is not a modelling parameter. It encodes what the two errors cost,
  and that is your judgement to make and defend.

---

### Optional capstone

You will be given a held-out dataset from a second pathosystem. In 20 minutes, build a
model and present one slide covering: what you predicted, how you split the data, what
baseline you beat, and one thing you would **not** claim from this model.